# MTP training on Kaggle

**Settings (right sidebar):** Accelerator **GPU T4 x2** (each GPU trains one run at a time), Internet **On**. No secrets needed (public repo, public model and data).

**How to run overnight**
1. Edit `GPU_QUEUES` in cell 3 (which configs, in which order, on which GPU).
2. Run cells 1-3 interactively and check: `git log` shows the commit you expect, pytest says `N passed`, and after ~7 min both runs print `step 20 | ... s/step` (T4 x2: ~0.34 s/step).
3. Stop the interactive session (power icon), then **Save Version → Save & Run All (Commit)**. Cell 4 keeps the session alive until every run finishes. Everything under `/kaggle/working` (runs/, logs) becomes the version's output.

**Resume / continue in a new session:** *Add Input →* the previous version's output. Cell 2 copies its `runs/` back and `train.py --resume auto` continues from the last saved step (every 500 steps, and on exit).

**Publish a run for evaluation (Om):** *New Dataset* from the version output, named `mtp-run-<ID>`.

In [ ]:
# 1. clone + install + tests
%cd /kaggle/working
!pkill -f "[s]cripts/train.py"; sleep 2
!rm -rf /tmp/mtp && git clone -q --depth 1 https://github.com/rampuriajainam/mtp-indic-wordgroup.git /tmp/mtp
%cd /tmp/mtp
!git log -1 --oneline
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10; peft 0.21 refuses LoRA when it is present
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests 2>&1 | tail -2

In [ ]:
# 2. restore runs from an attached earlier version (no-op if nothing is attached)
import glob, os, shutil
os.makedirs("/kaggle/working/runs", exist_ok=True)
for cfg_path in glob.glob("/kaggle/input/**/runs/*/config.yaml", recursive=True):
    src = os.path.dirname(cfg_path)
    dst = os.path.join("/kaggle/working/runs", os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print("restored", dst)

In [ ]:
# 3. parameters, then start training in the background and check after 7 minutes
GPU_QUEUES = {0: ["R2", "R1"], 1: ["R0"]}   # configs/<name>.yaml; each GPU runs its list in order
EXTRA_ARGS = ""                             # e.g. "--set data.eval_split=eval"
BUDGET_MIN = 630                            # Kaggle stops at 12 h; runs save and exit before this

import subprocess, time
lines = ["cd /tmp/mtp", "export PYTHONUTF8=1", "START=$(date +%s)", "run() {",
         "  LEFT=$(( %d - ($(date +%%s) - START) / 60 ))" % BUDGET_MIN,
         '  if [ $LEFT -lt 45 ]; then echo "skip $2 ($LEFT min left)"; return; fi',
         '  echo "start $2 on GPU $1 ($LEFT min left)"',
         "  CUDA_VISIBLE_DEVICES=$1 python -u scripts/train.py --config configs/$2.yaml --run_root /kaggle/working/runs"
         " --resume auto --stop_after_min $LEFT %s >> /kaggle/working/train_$2.log 2>&1" % EXTRA_ARGS,
         '  echo "done $2 (exit $?)"', "}"]
for gpu, queue in GPU_QUEUES.items():
    lines.append("(" + "; ".join(f"run {gpu} {name}" for name in queue) + ") &")
lines.append("wait")
open("/tmp/run_all.sh", "w").write("\n".join(lines) + "\n")

proc = subprocess.Popen(["bash", "/tmp/run_all.sh"], stdout=open("/kaggle/working/run_all.out", "w"),
                 stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(420)
!cat /kaggle/working/run_all.out
!tail -n 3 /kaggle/working/train_*.log

In [ ]:
# 4. keep the committed session alive until all runs finish, then summarise
print('waiting for training to finish ...', flush=True)
print('run_all.sh exit code', proc.wait())   # waits on the process itself (pgrep -f would match its own shell)
!cat /kaggle/working/run_all.out
!grep -h -A1 "eval step" /kaggle/working/train_*.log | tail -n 24